# 🚀 GIAI ĐOẠN 5 — KIẾN TRÚC STAIR5-v5.2 / NLGCL-BPE: KNOWN-POSITIVE EXCLUSION TRÊN BACKBONE NLGCL-CSE
### 🏆 Kaggle ML Engineering Pipeline — Multimodal Recommendation with Budgeted Path Expansion Contrastive Learning & Contraction-Preserving Adaptive Mixing

> **Đề tài:** Recommender Systems using Graph Representation: Multi-modal  
> **Khóa luận tốt nghiệp:** Khóa 2021–2025 — Khoa Công nghệ Thông tin, Trường Đại học Khoa học Tự nhiên, ĐHQG-HCM  
> **Sinh viên thực hiện:** Lê Hà Thanh Chương (23120195) & Bùi Trung Hiếu (23120257)  
> **Giảng viên hướng dẫn:** TS. Nguyễn Ngọc Thảo  
> **Kiến trúc đề xuất:** **STAIR5-v5.2 / NLGCL-BPE** (*Neighborhood-enriched Contrastive Learning with Budgeted Path Expansion*)
> **Tài liệu phương pháp luận & Thiết kế:** `docs/giai_doan_5/STAIR5_v5.1_Report.md` & `docs/giai_doan_5/STAIR5_v4_Experiment_Report.md`  
> **Mã nguồn lõi:** `models/stair5_v52.py`, `models/stair5_v52_objectives.py`, `models/stair5_v52_graph.py`, `models/stair5_v52_utils.py`, `optimizers/stair5_v52_smoother.py`, `main_stair5_v52.py`  
> **Tập dữ liệu mục tiêu:** **Amazon Sports**, **Amazon Baby**, **Amazon Electronics** (3 tập benchmark chuẩn)  
> **Chiến lược thực thi:** **3-Tier Stage Execution** (`Pha A: Sports [Pilot] ➔ Pha B: Baby [Confirmatory] ➔ Pha C: Electronics [Scale-up]`)
> **Đo lường VRAM:** Paper Standard (`torch.cuda.max_memory_allocated()`, trích xuất trực tiếp bằng cả 2 khóa `vram_allocated_mb` và `peak_allocated_gib * 1024` từ `training_telemetry.jsonl` và host NVML)

---

## 📑 1. BẢN THIẾT KẾ KIẾN TRÚC ĐỀ XUẤT: STAIR5-v5.2 / NLGCL-BPE

| Thành Phần Trụ Cột | Cơ Chế Kỹ Thuật STAIR5-v5.2 | Ý Nghĩa Khoa Học & Đột Phá So Với v4 |
|:---|:---|:---|
| **1. Kế thừa Backbone v4** | MI SVD Whitening, Tích chập phổ FSC, toán tử BSC Smoother trong AdamWSEvo, hàm mất mát BPR, scorer dot-product Euclidean $O(1)$. | Giữ nguyên vẹn độ trễ suy luận, tái lập chính xác v4 khi $\rho=0$ hoặc arm `V4`. |
| **2. Unique Entity Keys** | Deduplicate key entity IDs ($K_U, K_I$) trong batch, chuyển GEMM từ $B \times B$ sang $B \times m$. | Loại bỏ lãng phí tính toán trên các key trùng lặp, chuẩn hóa xác suất InfoNCE trên không gian thực thể duy nhất. |
| **3. Giữ Nguyên B Query Occurrences** | Không deduplicate query rows; giữ nguyên $B$ sampled query occurrences với trọng số công bằng. | Tránh hiện tượng méo phân phối bậc và không thiên lệch sang các entity phổ biến khi tính toán gradient. |
| **4. Budgeted Path Expansion (KPE)** | $A_{bk} = \mathbf{1}[k = t_b] \lor \neg M_{bk}$; loại triệt để các train-positive khác khỏi mẫu số InfoNCE của từng query. | Triệt tiêu xung đột gradient giả (false-negative repulsion), bảo vệ các mối quan hệ tích cực đã biết trong tập huấn luyện. |
| **5. GPU Membership Searchsorted** | Tra cứu quan hệ train-positive bằng `torch.searchsorted` trên tensor 1D int64 pair keys $(u \cdot n_I + i)$ trực tiếp trên GPU. | Hoàn toàn không dùng vòng lặp Python, không gọi `.item()` hay tạo ma trận dense $U \times I$. |
| **6. Checkpoint Anchor Chunking** | Tái tính toán logits và membership trong từng chunk $(C \times m)$ khi backward (non-reentrant checkpoint). | Giữ peak VRAM ở mức cực thấp, bảo đảm an toàn bộ nhớ tuyệt đối trên tập dữ liệu lớn Electronics ($B=4096$). |
| **7. Toán Tử CAM (Factorial Arm)** | $S_{\text{CAM}} = A S_0 A + B \bar{S}_{\text{CF}} B$ với $A=\text{diag}\sqrt{1-\eta}, B=\text{diag}\sqrt{\eta}$. | Bảo toàn chặn phổ $\|S_{\text{CAM}}\|_2 \le 1.0$ cho mọi $\delta$; khôi phục $S_4$ chính xác khi $\delta=0$. |

```
                ┌────────────────────────────────────────────────────────┐
                │            STAIR5-v5.2 / NLGCL-BPE ARCHITECTURE        │
                └────────────────────────────────────────────────────────┘
                                             │
      ┌──────────────────────────────────────┴──────────────────────────────────────┐
      ▼                                                                             ▼
[CALIBRATED S4 / CAM OPERATOR]                                            [RECOMMENDATION BACKBONE]
Modality Features ➔ Baseline kNN ➔ S0                                      User-Item Graph ➔ FSC Convolution
Train Interactions ➔ Block-wise Co-occurrence ➔ Shrinkage q_ij             Euclidean Dot-Product Scorer ➔ BPR
      ↓                                                                             ↓
Top-k CF Graph W_CF ➔ Normalized S_CF with Isolated Fallback               L_BPR Loss
      ↓                                                                             ↓
S4 = 0.9 S0 + 0.1 S_CF (or S_CAM with δ=0.05)                             Item Gradients Smoothed by S4 / CAM
(Sparse CSR, bounded ||S||_2 <= 1.0)                                                ↓
                                                                          [NLGCL-BPE Objective]
                                                                          Unique Entity Keys (m_U, m_I)
                                                                          Preserved B Query Occurrences
                                                                          Budgeted Path Expansion (KPE)
                                                                          L_total = L_BPR + λ_nlgcl * L_KPE
```


## Cell 1 ⚙️ Thiết lập Môi trường, Dependencies & Đồng bộ Mã Nguồn STAIR5-v5.2
- Đồng bộ mã nguồn mới nhất từ GitHub repository `ThanhChuong12/STAIR-Enhanced` (branch `main`).
- Cài đặt các gói phụ thuộc chuẩn tắc tương thích với môi trường nghiệm thu: `freerec>=0.9.7`, `torchdata==0.8.0`, `torch-geometric`, `nvidia-ml-py`, `prettytable`, `matplotlib`, `pyyaml`, `scipy`, `pandas`.
- Khởi chạy tiến trình con xác thực (`Subprocess Environment Verification`) để kiểm tra toàn bộ imports và GPU runtime.


In [ ]:
# Cell 1: Môi trường, Dependencies & Đồng bộ STAIR-Enhanced (STAIR5-v5.2 NLGCL-BPE)
import os, shutil, subprocess, sys, types, glob, torch
from pathlib import Path

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working') if os.path.exists('/kaggle/working') else None

# 1. Đồng bộ repository STAIR-Enhanced thông minh & an toàn
if os.path.exists(STAIR_DIR):
    print("Thư mục STAIR-Enhanced đã tồn tại. Đang kiểm tra đồng bộ mã nguồn...")
    try:
        subprocess.run(['git', '-C', STAIR_DIR, 'fetch', 'origin', 'main'], check=True, capture_output=True)
        chk = subprocess.run(['git', '-C', STAIR_DIR, 'ls-tree', 'origin/main', 'models/stair5_v52.py'], capture_output=True, text=True)
        if 'models/stair5_v52.py' in chk.stdout:
            subprocess.run(['git', '-C', STAIR_DIR, 'reset', '--hard', 'origin/main'], check=True)
            print("✅ Đã đồng bộ về commit mới nhất của origin/main (chứa STAIR5-v5.2 NLGCL-BPE).")
        else:
            print("⚠️ Remote origin/main chưa có models/stair5_v52.py. Giữ nguyên mã nguồn hiện có trên đĩa.")
    except Exception as e:
        print(f"⚠️ Cảnh báo git fetch/check ({e}). Tiếp tục dùng mã nguồn hiện có trên đĩa.")

if not os.path.exists(STAIR_DIR) and os.path.exists('/kaggle/working'):
    print("Cloning STAIR-Enhanced repository (branch main)...")
    try:
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/ThanhChuong12/STAIR-Enhanced.git', STAIR_DIR
        ], check=True)
    except Exception as e:
        print(f"⚠️ Git clone gặp cảnh báo: {e}")

# Fallback: Tự động dò quét và sao chép mã nguồn từ Kaggle Input nếu chưa có models/stair5_v52.py
if not os.path.exists(os.path.join(STAIR_DIR, 'models', 'stair5_v52.py')):
    input_matches = glob.glob('/kaggle/input/**/models/stair5_v52.py', recursive=True)
    if input_matches:
        src_repo = os.path.dirname(os.path.dirname(input_matches[0]))
        print(f"📦 Phát hiện mã nguồn STAIR5-v5.2 từ Kaggle Input: {src_repo} -> Sao chép vào {STAIR_DIR}...")
        os.makedirs(STAIR_DIR, exist_ok=True)
        shutil.copytree(src_repo, STAIR_DIR, dirs_exist_ok=True)

active_dir = STAIR_DIR if os.path.exists(STAIR_DIR) else os.path.abspath('.')
for p in [active_dir, STAIR_DIR, '/kaggle/working', '.']:
    if p and os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

if os.path.exists(STAIR_DIR):
    os.chdir(STAIR_DIR)

# Thiết lập biến môi trường chuẩn deterministic và PYTHONPATH cho subprocess
sub_env = os.environ.copy()
sub_env['PYTHONPATH'] = f"{active_dir}:{os.environ.get('PYTHONPATH', '')}"
sub_env['PYTHONWARNINGS'] = 'ignore'
sub_env.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['PYTHONPATH'] = sub_env['PYTHONPATH']
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# Xóa cache module để kernel luôn nạp phiên bản v5.1 mới nhất từ đĩa
for mod_name in list(sys.modules.keys()):
    if any(k in mod_name for k in ['stair5_v52', 'models.stair5_v52', 'optimizers.stair5_v52_smoother']):
        sys.modules.pop(mod_name, None)

# 2. Cài đặt các gói phụ thuộc chuẩn tắc tương thích với môi trường nghiệm thu
print("Cài đặt dependencies tương thích (freerec, torchdata, torch-geometric, pyyaml, prettytable)...")
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'pynvml', 'torchdata'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchdata==0.8.0'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'freerec>=0.9.7', 'torch-geometric', 'nvidia-ml-py', 'prettytable', 'matplotlib', 'pyyaml', 'scipy', 'pandas'
], check=False)

# 3. Patch tương thích nội bộ PyTorch Dynamo & TorchData DataPipe Registration
try:
    import models.freerec_compat
except Exception:
    try:
        import freerec_compat
    except Exception:
        pass

# 4. Xác minh môi trường trong tiến trình con (Subprocess Verification)
verify_script = '''
import torch, sys
print("=== SUBPROCESS ENVIRONMENT VERIFICATION ===")
print("Python Executable :", sys.executable)
print("Python Version    :", sys.version.split()[0])
print("PyTorch Version   :", torch.__version__)
print("CUDA Available    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name       :", torch.cuda.get_device_name(0))
    print("Allocated VRAM    : {:.2f} MB".format(torch.cuda.memory_allocated() / (1024**2)))
try:
    import freerec
    print("FreeRec Version   :", freerec.__version__)
except Exception as e:
    print("FreeRec Import    : FAILED ({})".format(e))
try:
    from models.stair5_v52 import STAIR5_v52_Model, ARMS_V52
    print("STAIR5-v5.2 Arms  :", list(ARMS_V52))
    print("STAIR5-v5.2 Core  : VERIFIED OK")
except Exception as e:
    print("STAIR5-v5.2 Core  : FAILED ({})".format(e))
print("===========================================")
'''
res = subprocess.run([sys.executable, '-c', verify_script], capture_output=True, text=True, env=sub_env)
print(res.stdout)
if res.stderr and "warning" not in res.stderr.lower():
    print("Stderr:", res.stderr)
print("✅ Khởi tạo môi trường STAIR5-v5.2 hoàn tất thành công.")


## Cell 2 📂 Chuẩn bị Dữ liệu từ Kaggle Input (Multi-Bridge sang /kaggle/data & Processed)
- Tự động dò quét toàn bộ kho dữ liệu Kaggle Input (`/kaggle/input`) để tìm kiếm 3 tập chuẩn: **Amazon Sports**, **Amazon Baby**, **Amazon Electronics**.
- Thiết lập cơ chế Symlink/Hardlink liên kết đa hướng sang `/kaggle/data`, `/kaggle/data/Processed`, `STAIR-Enhanced/data` để FreeRec luôn tìm thấy dữ liệu ở mọi vị trí.


In [ ]:
# Cell 2: Chuẩn bị dữ liệu từ Kaggle Input sang /kaggle/data & Processed
import os, shutil, glob

DATA_ROOT = '/kaggle/data'
PROCESSED_ROOT = os.path.join(DATA_ROOT, 'Processed')
LOCAL_DATA = '/kaggle/working/STAIR-Enhanced/data'
LOCAL_PROCESSED = os.path.join(LOCAL_DATA, 'Processed')

for d in [DATA_ROOT, PROCESSED_ROOT, LOCAL_DATA, LOCAL_PROCESSED]:
    os.makedirs(d, exist_ok=True)

TARGET_DATASETS = {
    'sports':      ('Amazon2014Sports_550_MMRec', ['sport', 'sports', 'amazon2014sports']),
    'baby':        ('Amazon2014Baby_550_MMRec', ['baby', 'amazon2014baby']),
    'electronics': ('Amazon2014Electronics_550_MMRec', ['electronic', 'electronics', 'amazon2014electronics']),
}

REQUIRED_EXTENSIONS = ('.npy', '.pkl', '.txt', '.inter', '.item', '.pt', '.csv', '.yaml')

def bridge_directories(src_dir, target_folder):
    '''Đồng bộ dữ liệu sang toàn bộ các vị trí FreeRec có thể tìm kiếm'''
    destinations = [
        os.path.join(DATA_ROOT, target_folder),
        os.path.join(PROCESSED_ROOT, target_folder),
        os.path.join(LOCAL_DATA, target_folder),
        os.path.join(LOCAL_PROCESSED, target_folder),
    ]
    for dst in destinations:
        if os.path.abspath(src_dir) == os.path.abspath(dst):
            continue
        os.makedirs(dst, exist_ok=True)
        for item in os.listdir(src_dir):
            s_item = os.path.join(src_dir, item)
            d_item = os.path.join(dst, item)
            if os.path.isdir(s_item):
                if not os.path.exists(d_item):
                    try:
                        os.symlink(s_item, d_item)
                    except Exception:
                        shutil.copytree(s_item, d_item, dirs_exist_ok=True)
            else:
                if not os.path.exists(d_item) or os.path.getsize(d_item) == 0:
                    try:
                        os.symlink(s_item, d_item)
                    except Exception:
                        shutil.copy2(s_item, d_item)

print("=" * 80)
print("TIẾN TRÌNH DÒ QUÉT & LIÊN KẾT DỮ LIỆU TỰ ĐỘNG CHO STAIR5-v5.2 NLGCL-BPE:")
prepared_data = set()
search_bases = ['/kaggle/input', '.', '..', '/kaggle/working']

for key, (folder_name, aliases) in TARGET_DATASETS.items():
    found_src = None
    for base in search_bases:
        if not os.path.exists(base):
            continue
        for root, dirs, files in os.walk(base):
            bname = os.path.basename(root).lower()
            if bname == folder_name.lower() or any(alias in bname for alias in aliases):
                has_req = any(f.endswith(REQUIRED_EXTENSIONS) for f in files)
                if has_req:
                    found_src = root
                    break
        if found_src:
            break

    if found_src:
        bridge_directories(found_src, folder_name)
        prepared_data.add(key)
        item_count = len(os.listdir(found_src))
        print(f"  ✅ [SẴN SÀNG] {key.upper():12s} -> Nguồn: {found_src} ({item_count} files)")
    else:
        print(f"  ⚠️ [CHƯA THẤY] {key.upper():12s} -> Sẽ nạp tự động qua kịch bản hoặc config.")
print("=" * 80)


## Cell 3 🧪 Bộ Kiểm Thử Độc Lập Toàn Diện Cho STAIR5-v5.2 NLGCL-BPE (9 Unit Tests)
- Thực thi nghiệm thu toán học và pipeline độc lập trước khi huấn luyện.
- Kiểm tra định danh tương đương Grouped Reference (Eq. 7), gradient của logit bị loại trừ triệt để bằng 0, trường hợp zero admissible negatives cho tổn thất 0 và gradient hữu hạn, tính tương đương gradient khi chunking, và chặn phổ co tự nhiên CAM $\|S_{\text{CAM}}\|_2 \le 1.0$.


In [ ]:
# Cell 3: Chạy Bộ Kiểm Thử Độc Lập Toàn Diện Cho STAIR5-v5.2 NLGCL-BPE
import os, sys, subprocess

print("=" * 80)
print("🚀 CHẠY BỘ KIỂM THỬ ĐỘC LẬP TOÀN DIỆN CHO STAIR5-v5.2 NLGCL-BPE (9 UNIT TESTS)...")
print("=" * 80)

test_cmd = [
    sys.executable, '-m', 'pytest',
    'tests/test_stair5_v52_objectives.py',
    'tests/test_stair5_v52_graph.py',
    'tests/test_stair5_v52_pipeline.py',
    '-v'
]

sub_env = os.environ.copy()
sub_env['PYTHONPATH'] = f".:{os.environ.get('PYTHONPATH', '')}"

res = subprocess.run(test_cmd, capture_output=True, text=True, env=sub_env)
print(res.stdout)
if res.returncode == 0:
    print("🎉 TẤT CẢ 9/9 UNIT TESTS CỦA STAIR5-v5.2 ĐÃ VƯỢT QUA XUẤT SẮC! SẴN SÀNG HUẤN LUYỆN.")
else:
    print(res.stderr)
    raise RuntimeError(f"Kiểm thử tiền trạm thất bại với mã lỗi {res.returncode}; dừng huấn luyện để bảo vệ tài nguyên.")
print("=" * 80)


## Cell 4 🛠️ Telemetry Engine: Training Runner, GPU VRAM Profiler & Visualizer
- Quản lý tiến trình huấn luyện `main_stair5_v52.py` trong sub-process an toàn, cô lập hoàn toàn bộ nhớ.
- Giám sát VRAM phần cứng thời gian thực qua background thread (`nvidia-ml-py` / NVML).
- Trích xuất tự động `training_telemetry.jsonl` và checkpoint tốt nhất (`[Coach] >>> Load best model`).
- Cung cấp các hàm vẽ hồ sơ VRAM (`plot_single_dataset_vram`) và đường cong hội tụ (`plot_single_dataset_learning_curves`) chuyên sâu cho từng pha.


In [ ]:
# Cell 4: Telemetry Engine — Training Runner, Hardware Profiler & Visualization
import os, sys, time, re, json, threading, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.stdout.reconfigure(encoding='utf-8') if hasattr(sys.stdout, 'reconfigure') else None

TRACKED_METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

# Đối chuẩn đa thế hệ lịch sử trên cả 3 tập dữ liệu
BASELINE_REF = {
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V4_REF = {
    'sports':      {'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
    'baby':        {'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
    'electronics': {'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
}

DATASET_PROFILES = {
    'sports': {
        'name':        'Amazon Sports',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '35,598 Users | 18,357 Items | 296K Interactions',
        'color':       '#ff7f0e',
        'approx_mins': 42.0,
    },
    'baby': {
        'name':        'Amazon Baby',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '19,445 Users | 7,050 Items | 160K Interactions',
        'color':       '#1f77b4',
        'approx_mins': 20.0,
    },
    'electronics': {
        'name':        'Amazon Electronics',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '192,403 Users | 63,001 Items | 1.69M Interactions',
        'color':       '#2ca02c',
        'approx_mins': 345.0,
    },
}

vram_profile = {}

def vram_monitor(key, stop_evt, interval=2.0):
    '''Background thread tracking host GPU allocated memory via NVML.'''
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        records = []
        while not stop_evt.is_set():
            mem = pynvml.nvmlDeviceGetMemoryInfo(h)
            records.append(mem.used / (1024**2))
            time.sleep(interval)
        pynvml.nvmlShutdown()
        vram_profile[key] = records
    except Exception:
        vram_profile[key] = []

def resolve_path(p):
    if os.path.exists(p):
        return p
    base = os.path.basename(p)
    candidates = [
        p,
        os.path.join('/kaggle/working/logs/stair5_v52', base),
        os.path.join('/kaggle/working/logs/GD5', base),
        os.path.join('logs/stair5_v52', base),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return p

def parse_telemetry_jsonl(artifact_dir):
    p = Path(artifact_dir) / "training_telemetry.jsonl"
    if not p.is_file():
        return []
    records, seen = [], set()
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
                ep = data.get("epoch")
                if ep not in seen:
                    seen.add(ep)
                    records.append(data)
            except Exception:
                continue
    return records

def extract_best_validation(log_path):
    '''Extracts the epoch and metrics of the best validation checkpoint.'''
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    matches = re.findall(r'VALID\s+@Epoch:\s*(\d+).*?NDCG@20\s+Avg:\s*([0-9.]+)', content, re.DOTALL)
    if not matches:
        return None, {}
    best_ep, best_val = -1, -1.0
    for ep_str, val_str in matches:
        ep, val = int(ep_str), float(val_str)
        if val > best_val:
            best_val = val
            best_ep = ep
    return best_ep, {'NDCG@20': best_val}

def extract_test_metrics(log_path):
    '''Extracts final test metrics corresponding to the best validation checkpoint.'''
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    best_matches = list(re.finditer(r'\[Coach\]\s*>>>\s*Load best model\s*@Epoch:\s*(\d+)', content))
    best_match = best_matches[-1] if best_matches else None
    if not best_match:
        return None, {}
    best_model_epoch = int(best_match.group(1))
    content_after = content[best_match.end():]
    test_match = re.search(r'TEST\s+@Epoch:\s*(\d+)(.*?)(?:=================|$)', content_after, re.DOTALL)
    if not test_match:
        return None, {}
    test_epoch = int(test_match.group(1))
    test_snippet = test_match.group(2)
    test_metrics = {}
    for metric in TRACKED_METRICS:
        m = re.search(rf'{metric}(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', test_snippet, re.IGNORECASE)
        if m:
            test_metrics[metric] = float(m.group(1))
    return test_epoch, test_metrics

def parse_training_losses(log_path, telemetry=None):
    '''Extracts BPR loss, KPE CL loss, and Total loss trajectories.'''
    if telemetry and len(telemetry) > 0:
        bpr_losses = [(r['epoch'], float(r['bpr_loss'])) for r in telemetry if 'bpr_loss' in r]
        cl_losses = [(r['epoch'], float(r['nlgcl_loss'])) for r in telemetry if 'nlgcl_loss' in r]
        total_losses = [(r['epoch'], float(r['loss'])) for r in telemetry if 'loss' in r]
        return bpr_losses, cl_losses, total_losses
    log_path = resolve_path(log_path)
    bpr_losses, cl_losses, total_losses = [], [], []
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        telemetry_matches = re.findall(
            r'\[Epoch\s+(\d+)\s+Telemetry\]\s+Loss:\s*([0-9.]+)\s+\|\s+BPR:\s*([0-9.]+)\s+\|\s+CL:\s*([0-9.]+)',
            content
        )
        if telemetry_matches:
            for ep, l, bpr, cl in telemetry_matches:
                total_losses.append((int(ep), float(l)))
                bpr_losses.append((int(ep), float(bpr)))
                cl_losses.append((int(ep), float(cl)))
    return bpr_losses, cl_losses, total_losses

def parse_valid_metric(log_path, metric='NDCG@20'):
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    pattern = rf'VALID\s+@Epoch:\s*(\d+).*?{metric}\s+Avg:\s*([0-9.]+)'
    matches = re.findall(pattern, content, re.IGNORECASE)
    return [(int(ep), float(v)) for ep, v in matches]

def run_training_stair5_v52(key, config_yaml=None, data_root='/kaggle/data', log_path=None, artifact_dir=None, **kwargs):
    '''Executes STAIR5-v5.2 NLGCL-BPE training with CLI argument overrides and strict error handling.'''
    cfg_dict = STAIR5_V52_CONFIGS.get(key, {}).copy()
    cfg_dict.update(kwargs)

    config_yaml = config_yaml or cfg_dict.get('yaml')
    label = f"{key}_{cfg_dict.get('v52_arm', 'P-BPE')}_seed{cfg_dict.get('seed', 1)}"
    log_path = log_path or str(Path(cfg_dict['log']).parent / (label + '.log'))
    artifact_dir = artifact_dir or str(Path(cfg_dict['artifact_dir']).parent / label)
    resuming = bool(cfg_dict.get('resume_from'))
    if not resuming and (Path(log_path).exists() or Path(artifact_dir).exists()):
        print(f'⚠️ Thông báo: Ghi đè hoặc tái lập kịch bản run cho {label}.')
        if Path(artifact_dir).exists():
            for stale in ('manifest.json', 'resume_manifest.json', 'training_checkpoint.pt', 'training_telemetry.jsonl', 'best_model.pt', 'last_model.pt', 'diagnostics_epoch1.json'):
                sp = Path(artifact_dir) / stale
                if sp.exists():
                    try:
                        sp.unlink()
                    except OSError:
                        pass
    STAIR5_V52_CONFIGS[key].update(log=log_path, artifact_dir=artifact_dir, v52_arm=cfg_dict.get('v52_arm', 'P-BPE'))

    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    os.makedirs(artifact_dir, exist_ok=True)

    print('=' * 80)
    print(f'🚀 KHỞI ĐỘNG TIẾN TRÌNH HUẤN LUYỆN STAIR5-v5.2 / NLGCL-BPE: {key.upper()}')
    print(f'  * Dataset Key         : {key}')
    print(f'  * YAML Configuration  : {config_yaml}')
    print(f'  * Log Path            : {log_path}')
    print(f'  * Artifact Directory  : {artifact_dir}')
    print(f'  * V5.1 Arm            : {cfg_dict.get("v52_arm", "P-BPE")}')
    print(f'  * NLGCL Weight (λ)    : {cfg_dict.get("lambda_nlgcl", 0.01)}')
    print(f'  * NLGCL Temp (τ)      : {cfg_dict.get("nlgcl_tau", 0.2)}')
    print(f'  * NLGCL Rho (ρ)       : {cfg_dict.get("nlgcl_rho", 1.0)}')
    print(f'  * Operator Blend (η)  : {cfg_dict.get("eta", 0.1)}')
    print(f'  * Seed                : {cfg_dict.get("seed", 1)}')
    print('=' * 80)

    stop_evt = threading.Event()
    th = threading.Thread(target=vram_monitor, args=(key, stop_evt), daemon=True)
    th.start()

    t0 = time.time()
    runner_py = '/kaggle/working/STAIR-Enhanced/main_stair5_v52.py'
    if not os.path.exists(runner_py):
        runner_py = 'main_stair5_v52.py'

    if not os.path.exists(config_yaml):
        cand_y = os.path.join('configs', os.path.basename(config_yaml))
        if os.path.exists(cand_y):
            config_yaml = cand_y

    device_str = str(cfg_dict.get('device', '0' if torch.cuda.is_available() else 'cpu'))
    cmd = [sys.executable, '-u', str(runner_py), '--config', str(config_yaml),
           '--root', str(data_root), '--device', str(device_str), '--ranking', 'full',
           '--artifact-dir', str(artifact_dir)]
    defaults = {'v52_arm': 'P-BPE', 'seed': 1, 'eval_freq': 5, 'eta': 0.1,
                'k_cf': 5, 'c_min': 2, 't_shrinkage': 5.0, 'cam_delta': 0.05,
                'cf_block_size': 64, 'cf_memory_budget_mib': 128,
                'lambda_nlgcl': .01, 'nlgcl_tau': .2, 'nlgcl_G': 1,
                'nlgcl_alpha': .5, 'nlgcl_rho': 1.0, 'cl_chunk_size': 1024,
                'knn_chunk_size': 1024, 'knn_device': 'auto',
                'graph_cache_dir': '/kaggle/working/runs/stair5_v52/graph_cache'}
    settings = {**defaults, **cfg_dict}
    for k_item in defaults:
        cmd += ['--' + k_item.replace('_', '-'), str(settings[k_item])]
    for k_item in ('epochs', 'batch_size', 'gamma', 'lr', 'weight_decay', 'num_workers'):
        if k_item in cfg_dict:
            cmd += ['--' + k_item.replace('_', '-'), str(cfg_dict[k_item])]
    if cfg_dict.get('resume_from'):
        cmd += ['--resume-from', str(cfg_dict['resume_from'])]

    print('Command:', ' '.join(cmd))
    env = os.environ.copy()
    env['PYTHONPATH'] = f".:{os.environ.get('PYTHONPATH', '')}"
    env['PYTHONUTF8'] = '1'

    try:
        with open(log_path, 'a' if resuming else 'w', encoding='utf-8') as lf:
            proc = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding='utf-8', errors='replace',
                bufsize=1,
                env=env,
            )
            for line in proc.stdout:
                sys.stdout.write(line)
                sys.stdout.flush()
                lf.write(line)
                lf.flush()
            proc.wait()
            ret_code = proc.returncode
    finally:
        stop_evt.set()
        th.join(timeout=3.0)

    elapsed = time.time() - t0
    print(f'⏱️ Thời gian thực thi ({key.upper()}): {elapsed/60:.2f} phút | Trạng thái: {ret_code}')
    if ret_code != 0:
        raise RuntimeError(f'Tiến trình huấn luyện {key.upper()} thất bại với mã lỗi {ret_code}. Kiểm tra log: {log_path}')
    return elapsed


## Cell 5 📋 Cấu hình Siêu tham số STAIR5-v5.2 NLGCL-BPE (Dataset-Adaptive Matrix)
- Ma trận cấu hình chi tiết cho 3 tập dữ liệu theo đặc tả toán học trong `docs/giai_doan_5/STAIR5_v5.1_Report.md`:
  * **Amazon Sports**: $\gamma = 0.2, wd = 0.1, \text{lr} = 0.001, \eta = 0.1, k_{\text{CF}} = 5, \lambda_{\text{nlgcl}} = 0.01, \tau=0.2, \alpha=0.5, \rho=1.0, \text{chunk}=1024$.
  * **Amazon Baby**: $\gamma = 0.1, wd = 0.3, \text{lr} = 0.001, \eta = 0.1, k_{\text{CF}} = 5, \lambda_{\text{nlgcl}} = 0.01, \tau=0.2, \alpha=0.5, \rho=1.0, \text{chunk}=1024$.
  * **Amazon Electronics**: $\gamma = 0.4, wd = 0.1, \text{lr} = 0.001, \text{batch\_size} = 4096, \eta = 0.1, k_{\text{CF}} = 5, \lambda_{\text{nlgcl}} = 0.01, \rho=1.0, \text{chunk}=512$.


In [ ]:
# Cell 5: Cấu hình Siêu tham số STAIR5-v5.2 NLGCL-BPE (Dataset-Adaptive Matrix)
import os, uuid
from datetime import datetime
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]

os.makedirs('/kaggle/working/logs/stair5_v52', exist_ok=True)
os.makedirs('/kaggle/working/reports', exist_ok=True)
os.makedirs('/kaggle/working/runs/stair5_v52/graph_cache', exist_ok=True)

STAIR5_V52_CONFIGS = {
    'sports': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Sports_STAIR5_v52.yaml',
        'log':                f'/kaggle/working/logs/stair5_v52/{RUN_ID}/sports_stair5_v52.log',
        'artifact_dir':       f'/kaggle/working/runs/stair5_v52/{RUN_ID}/sports_P-BPE_seed1',
        'v52_arm':            'P-BPE',
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cam_delta':          0.05,
        'cf_block_size':      64,
        'lambda_nlgcl':       0.01,
        'nlgcl_tau':          0.2,
        'nlgcl_G':            1,
        'nlgcl_alpha':        0.5,
        'nlgcl_rho':          1.0,
        'cl_chunk_size':      1024,
        'seed':               1,
        'eval_freq':          5,
    },
    'baby': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Baby_STAIR5_v52.yaml',
        'log':                f'/kaggle/working/logs/stair5_v52/{RUN_ID}/baby_stair5_v52.log',
        'artifact_dir':       f'/kaggle/working/runs/stair5_v52/{RUN_ID}/baby_P-BPE_seed1',
        'v52_arm':            'P-BPE',
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cam_delta':          0.05,
        'cf_block_size':      64,
        'lambda_nlgcl':       0.01,
        'nlgcl_tau':          0.2,
        'nlgcl_G':            1,
        'nlgcl_alpha':        0.5,
        'nlgcl_rho':          1.0,
        'cl_chunk_size':      1024,
        'seed':               1,
        'eval_freq':          5,
    },
    'electronics': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Electronics_STAIR5_v52.yaml',
        'log':                f'/kaggle/working/logs/stair5_v52/{RUN_ID}/electronics_stair5_v52.log',
        'artifact_dir':       f'/kaggle/working/runs/stair5_v52/{RUN_ID}/electronics_P-BPE_seed1',
        'v52_arm':            'P-BPE',
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cam_delta':          0.05,
        'cf_block_size':      64,
        'lambda_nlgcl':       0.01,
        'nlgcl_tau':          0.2,
        'nlgcl_G':            1,
        'nlgcl_alpha':        0.5,
        'nlgcl_rho':          1.0,
        'cl_chunk_size':      512,
        'seed':               1,
        'eval_freq':          5,
    },
}

print('✅ Ma trận cấu hình STAIR5-v5.2 NLGCL-BPE đã nạp thành công:')
for k, v in STAIR5_V52_CONFIGS.items():
    print(f"  * {k.upper():12s} -> Arm: {v['v52_arm']} | Config: {os.path.basename(v['yaml'])} | Chunk: {v['cl_chunk_size']}")


## Cell 6a 🏋️ Huấn luyện Pha A — Amazon Sports (Pilot & Đề Xuất Chính P-BPE)
- Chạy huấn luyện mô hình STAIR5-v5.2 trên tập **Amazon Sports** (35,598 Users, 18,357 Items, 296K Interactions).
- Áp dụng cơ chế Budgeted Path Expansion (KPE): loại các train-positive khác khỏi denominator của từng query occurrence.
- Ghi nhận telemetry từng epoch: BPR loss, KPE CL loss, VRAM phân bổ, và thời gian thực thi.


In [ ]:
# Cell 6a: Training STAIR5-v5.2 NLGCL-BPE on Amazon Sports (Pha A: Pilot)
sports_time = run_training_stair5_v52('sports')
ep_sports, metrics_sports = extract_test_metrics(STAIR5_V52_CONFIGS['sports']['log'])
val_ep_sports, val_m_sports = extract_best_validation(STAIR5_V52_CONFIGS['sports']['log'])

print('\n' + '=' * 80)
print('📊 KẾT QUẢ THỰC NGHIỆM PHA A — AMAZON SPORTS (STAIR5-v5.2 / P-BPE):')
print(f'  * Checkpoint tối ưu tại Epoch: {ep_sports} (Validation NDCG@20 = {val_m_sports.get("NDCG@20", 0.0):.4f})')
for m in TRACKED_METRICS:
    val = metrics_sports.get(m, 0.0)
    base = BASELINE_REF['sports'].get(m, 0.0)
    v4_val = V4_REF['sports'].get(m, 0.0)
    d_base = ((val - base) / base) * 100 if base > 0 else 0.0
    d_v4 = ((val - v4_val) / v4_val) * 100 if v4_val > 0 else 0.0
    print(f'  * {m:10s} : {val:.4f}  (vs Baseline: {d_base:+.2f}% | vs v4: {d_v4:+.2f}%)')
print('=' * 80)


## Cell 6b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Sports
- Trích xuất chuẩn xác mức tiêu thụ VRAM thực tế từ `training_telemetry.jsonl` bằng cả hai khóa `vram_allocated_mb` và `peak_allocated_gib * 1024`.
- Hiển thị mức đỉnh VRAM tensor và kiểm định tính ổn định qua 500 epochs.


In [ ]:
# Cell 6b: Model Tensor VRAM Profile — Amazon Sports (Paper Standard)
sports_art = STAIR5_V52_CONFIGS['sports']['artifact_dir']
telemetry_sports = parse_telemetry_jsonl(sports_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if telemetry_sports:
    eps = [r['epoch'] for r in telemetry_sports]
    vram_mb = [r.get('vram_allocated_mb', r.get('peak_allocated_gib', 0.0) * 1024) for r in telemetry_sports]
    peak_val = max(vram_mb)
    plt.plot(eps, vram_mb, color='#ff7f0e', lw=2.0, label=f'Peak Allocated Tensor VRAM (Observed: {peak_val:.1f} MiB)')
    plt.axhline(peak_val, color='#111111', ls='--', lw=1.2, label=f'Peak Tensor Memory: {peak_val:.1f} MiB ({peak_val/1024:.2f} GiB)')
    plt.title('Amazon Sports — Peak Model Tensor VRAM Profile (STAIR5-v5.2 NLGCL-BPE)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MiB)', fontsize=10)
    plt.ylim(0, max(peak_val * 1.3, 100))
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(frameon=True, loc='lower right')
elif 'sports' in vram_profile and len(vram_profile['sports']) > 0:
    plt.plot(vram_profile['sports'], color='#ff7f0e', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Sports — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(frameon=True)
else:
    plt.text(0.5, 0.5, 'Chưa có dữ liệu VRAM hoặc NVML không khả dụng', ha='center', va='center')

plt.tight_layout()
os.makedirs('/kaggle/working/reports', exist_ok=True)
plt.savefig('/kaggle/working/reports/vram_profile_sports.png', dpi=300)
plt.show()
print("✅ [Model Tensor VRAM Profile Saved] -> /kaggle/working/reports/vram_profile_sports.png")


## Cell 6c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Sports
- Trực quan hóa phân tách các thành phần tổn thất: Total Loss, Pure BPR Loss, và KPE Contrastive Loss.
- Trục phụ (Twinx) hiển thị quỹ đạo cải thiện Validation NDCG@20 đối chiếu trực tiếp với ngưỡng STAIR Baseline gốc và STAIR5-v4.


In [ ]:
# Cell 6c: Learning Dynamics & Convergence Profiles — Amazon Sports
fig, ax1 = plt.subplots(figsize=(10, 4.8), dpi=150)

log_sports = STAIR5_V52_CONFIGS['sports']['log']
bpr_l, cl_l, tot_l = parse_training_losses(log_sports, telemetry_sports)
val_ndcg = parse_valid_metric(log_sports, 'NDCG@20')

if tot_l:
    x_tot, y_tot = zip(*tot_l)
    ax1.plot(x_tot, y_tot, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·KPE)')
if bpr_l:
    x_bpr, y_bpr = zip(*bpr_l)
    ax1.plot(x_bpr, y_bpr, color='#2ca02c', lw=1.5, ls='--', alpha=0.8, label='Pure BPR Loss')
if cl_l:
    x_cl, y_cl = zip(*cl_l)
    ax1.plot(x_cl, y_cl, color='#d62728', lw=1.2, ls=':', alpha=0.7, label='Raw KPE Loss')

ax1.set_xlabel('Epoch', fontsize=10)
ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1.grid(True, linestyle='--', alpha=0.4)

if val_ndcg:
    ax2 = ax1.twinx()
    x_val, y_val = zip(*val_ndcg)
    ax2.plot(x_val, y_val, color='#ff7f0e', lw=2.2, marker='o', ms=3, label='Validation NDCG@20')
    ax2.set_ylabel('Validation NDCG@20', color='#ff7f0e', fontsize=10)
    ax2.tick_params(axis='y', labelcolor='#ff7f0e')
    if 'sports' in BASELINE_REF:
        ax2.axhline(BASELINE_REF['sports']['NDCG@20'], color='gray', ls='--', lw=1.2, label='Baseline STAIR (0.0500)')
    if 'sports' in V4_REF:
        ax2.axhline(V4_REF['sports']['NDCG@20'], color='#9467bd', ls=':', lw=1.2, label='STAIR5-v4 (0.0517)')
    # Combine legends
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=8.5, framealpha=0.9)
else:
    ax1.legend(loc='upper right', fontsize=9.0)

plt.title('Amazon Sports — Learning Dynamics & Convergence (STAIR5-v5.2 NLGCL-BPE)', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_sports.png', dpi=300)
plt.show()
print("✅ [Learning Dynamics Saved] -> /kaggle/working/reports/learning_curve_sports.png")


## Cell 7a 🏋️ Huấn luyện Pha B — Amazon Baby (Kiểm chứng An toàn Over-smoothing)
- Chạy huấn luyện mô hình STAIR5-v5.2 trên tập **Amazon Baby** (19,445 Users, 7,050 Items, 160K Interactions).
- Kiểm tra tính ổn định trên tập mật độ thưa cao với siêu tham số $\gamma=0.1, wd=0.3$.
- Đánh giá tác động của việc loại bỏ False-Negative Repulsion lên các item thưa.


In [ ]:
# Cell 7a: Training STAIR5-v5.2 NLGCL-BPE on Amazon Baby (Pha B: Confirmatory)
baby_time = run_training_stair5_v52('baby')
ep_baby, metrics_baby = extract_test_metrics(STAIR5_V52_CONFIGS['baby']['log'])
val_ep_baby, val_m_baby = extract_best_validation(STAIR5_V52_CONFIGS['baby']['log'])

print('\n' + '=' * 80)
print('📊 KẾT QUẢ THỰC NGHIỆM PHA B — AMAZON BABY (STAIR5-v5.2 / P-BPE):')
print(f'  * Checkpoint tối ưu tại Epoch: {ep_baby} (Validation NDCG@20 = {val_m_baby.get("NDCG@20", 0.0):.4f})')
for m in TRACKED_METRICS:
    val = metrics_baby.get(m, 0.0)
    base = BASELINE_REF['baby'].get(m, 0.0)
    v4_val = V4_REF['baby'].get(m, 0.0)
    d_base = ((val - base) / base) * 100 if base > 0 else 0.0
    d_v4 = ((val - v4_val) / v4_val) * 100 if v4_val > 0 else 0.0
    print(f'  * {m:10s} : {val:.4f}  (vs Baseline: {d_base:+.2f}% | vs v4: {d_v4:+.2f}%)')
print('=' * 80)


## Cell 7b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Baby
- Trực quan hóa mức tiêu thụ VRAM thuần bộ nhớ tensor trong suốt quá trình huấn luyện Amazon Baby.


In [ ]:
# Cell 7b: Model Tensor VRAM Profile — Amazon Baby (Paper Standard)
baby_art = STAIR5_V52_CONFIGS['baby']['artifact_dir']
telemetry_baby = parse_telemetry_jsonl(baby_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if telemetry_baby:
    eps = [r['epoch'] for r in telemetry_baby]
    vram_mb = [r.get('vram_allocated_mb', r.get('peak_allocated_gib', 0.0) * 1024) for r in telemetry_baby]
    peak_val = max(vram_mb)
    plt.plot(eps, vram_mb, color='#1f77b4', lw=2.0, label=f'Peak Allocated Tensor VRAM (Observed: {peak_val:.1f} MiB)')
    plt.axhline(peak_val, color='#111111', ls='--', lw=1.2, label=f'Peak Tensor Memory: {peak_val:.1f} MiB ({peak_val/1024:.2f} GiB)')
    plt.title('Amazon Baby — Peak Model Tensor VRAM Profile (STAIR5-v5.2 NLGCL-BPE)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MiB)', fontsize=10)
    plt.ylim(0, max(peak_val * 1.3, 100))
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(frameon=True, loc='lower right')
elif 'baby' in vram_profile and len(vram_profile['baby']) > 0:
    plt.plot(vram_profile['baby'], color='#1f77b4', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Baby — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(frameon=True)
else:
    plt.text(0.5, 0.5, 'Chưa có dữ liệu VRAM hoặc NVML không khả dụng', ha='center', va='center')

plt.tight_layout()
plt.savefig('/kaggle/working/reports/vram_profile_baby.png', dpi=300)
plt.show()
print("✅ [Model Tensor VRAM Profile Saved] -> /kaggle/working/reports/vram_profile_baby.png")


## Cell 7c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Baby
- Trực quan hóa tiến trình huấn luyện STAIR5-v5.2 trên tập **Amazon Baby**: Phân tách Pure BPR Loss, KPE Loss, Total Loss và Validation NDCG@20.


In [ ]:
# Cell 7c: Learning Dynamics & Convergence Profiles — Amazon Baby
fig, ax1 = plt.subplots(figsize=(10, 4.8), dpi=150)

log_baby = STAIR5_V52_CONFIGS['baby']['log']
bpr_b, cl_b, tot_b = parse_training_losses(log_baby, telemetry_baby)
val_ndcg_b = parse_valid_metric(log_baby, 'NDCG@20')

if tot_b:
    x_tot, y_tot = zip(*tot_b)
    ax1.plot(x_tot, y_tot, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·KPE)')
if bpr_b:
    x_bpr, y_bpr = zip(*bpr_b)
    ax1.plot(x_bpr, y_bpr, color='#ff7f0e', lw=1.5, ls='--', alpha=0.8, label='Pure BPR Loss')
if cl_b:
    x_cl, y_cl = zip(*cl_b)
    ax1.plot(x_cl, y_cl, color='#d62728', lw=1.2, ls=':', alpha=0.7, label='Raw KPE Loss')

ax1.set_xlabel('Epoch', fontsize=10)
ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1.grid(True, linestyle='--', alpha=0.4)

if val_ndcg_b:
    ax2 = ax1.twinx()
    x_val, y_val = zip(*val_ndcg_b)
    ax2.plot(x_val, y_val, color='#2ca02c', lw=2.2, marker='s', ms=3, label='Validation NDCG@20')
    ax2.set_ylabel('Validation NDCG@20', color='#2ca02c', fontsize=10)
    ax2.tick_params(axis='y', labelcolor='#2ca02c')
    if 'baby' in BASELINE_REF:
        ax2.axhline(BASELINE_REF['baby']['NDCG@20'], color='gray', ls='--', lw=1.2, label='Baseline STAIR (0.0454)')
    if 'baby' in V4_REF:
        ax2.axhline(V4_REF['baby']['NDCG@20'], color='#9467bd', ls=':', lw=1.2, label='STAIR5-v4 (0.0461)')
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=8.5, framealpha=0.9)
else:
    ax1.legend(loc='upper right', fontsize=9.0)

plt.title('Amazon Baby — Learning Dynamics & Convergence (STAIR5-v5.2 NLGCL-BPE)', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_baby.png', dpi=300)
plt.show()
print("✅ [Learning Dynamics Saved] -> /kaggle/working/reports/learning_curve_baby.png")


## Cell 8a 🏋️ Huấn luyện Pha C — Amazon Electronics (Quy mô Lớn ~1.7M Tương tác)
- Kiểm chứng khả năng mở rộng (Scalability) của STAIR5-v5.2 trên tập dữ liệu lớn nhất **Amazon Electronics** (192,403 Users, 63,001 Items, 1.69M Interactions).
- Cấu hình tối ưu bộ nhớ: $\gamma = 0.4, \text{batch\_size} = 4096, \text{cl\_chunk\_size} = 512, \rho=1.0$.
- Cơ chế Anchor Chunking kết hợp Gradient Checkpointing bảo đảm an toàn VRAM < 1 GiB trên GPU Tesla T4.


In [ ]:
# # Cell 8a: Training STAIR5-v5.2 NLGCL-BPE on Amazon Electronics (Pha C: Scale-up)
# electronics_time = run_training_stair5_v52('electronics')
# ep_elec, metrics_elec = extract_test_metrics(STAIR5_V52_CONFIGS['electronics']['log'])
# val_ep_elec, val_m_elec = extract_best_validation(STAIR5_V52_CONFIGS['electronics']['log'])

# print('\n' + '=' * 80)
# print('📊 KẾT QUẢ THỰC NGHIỆM PHA C — AMAZON ELECTRONICS (STAIR5-v5.2 / P-BPE):')
# print(f'  * Checkpoint tối ưu tại Epoch: {ep_elec} (Validation NDCG@20 = {val_m_elec.get("NDCG@20", 0.0):.4f})')
# for m in TRACKED_METRICS:
#     val = metrics_elec.get(m, 0.0)
#     base = BASELINE_REF['electronics'].get(m, 0.0)
#     v4_val = V4_REF['electronics'].get(m, 0.0)
#     d_base = ((val - base) / base) * 100 if base > 0 else 0.0
#     d_v4 = ((val - v4_val) / v4_val) * 100 if v4_val > 0 else 0.0
#     print(f'  * {m:10s} : {val:.4f}  (vs Baseline: {d_base:+.2f}% | vs v4: {d_v4:+.2f}%)')
# print('=' * 80)


## Cell 8b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Electronics
- Trích xuất và kiểm tra hồ sơ bộ nhớ VRAM trên tập quy mô lớn Amazon Electronics.


In [ ]:
# # Cell 8b: Model Tensor VRAM Profile — Amazon Electronics (Paper Standard)
# elec_art = STAIR5_V52_CONFIGS['electronics']['artifact_dir']
# telemetry_elec = parse_telemetry_jsonl(elec_art)

# plt.figure(figsize=(9, 4.5), dpi=150)
# if telemetry_elec:
#     eps = [r['epoch'] for r in telemetry_elec]
#     vram_mb = [r.get('vram_allocated_mb', r.get('peak_allocated_gib', 0.0) * 1024) for r in telemetry_elec]
#     peak_val = max(vram_mb)
#     plt.plot(eps, vram_mb, color='#2ca02c', lw=2.0, label=f'Peak Allocated Tensor VRAM (Observed: {peak_val:.1f} MiB)')
#     plt.axhline(peak_val, color='#111111', ls='--', lw=1.2, label=f'Peak Tensor Memory: {peak_val:.1f} MiB ({peak_val/1024:.2f} GiB)')
#     plt.title('Amazon Electronics — Peak Model Tensor VRAM Profile (STAIR5-v5.2 NLGCL-BPE)', fontsize=12, fontweight='bold')
#     plt.xlabel('Epoch', fontsize=10)
#     plt.ylabel('Allocated VRAM (MiB)', fontsize=10)
#     plt.ylim(0, max(peak_val * 1.3, 100))
#     plt.grid(True, linestyle='--', alpha=0.4)
#     plt.legend(frameon=True, loc='lower right')
# elif 'electronics' in vram_profile and len(vram_profile['electronics']) > 0:
#     plt.plot(vram_profile['electronics'], color='#2ca02c', lw=1.8, label='Host NVML GPU Used (MB)')
#     plt.title('Amazon Electronics — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
#     plt.xlabel('Polling Step (~2s)', fontsize=10)
#     plt.ylabel('Memory (MB)', fontsize=10)
#     plt.grid(True, linestyle='--', alpha=0.4)
#     plt.legend(frameon=True)
# else:
#     plt.text(0.5, 0.5, 'Chưa có dữ liệu VRAM hoặc NVML không khả dụng', ha='center', va='center')

# plt.tight_layout()
# plt.savefig('/kaggle/working/reports/vram_profile_electronics.png', dpi=300)
# plt.show()
# print("✅ [Model Tensor VRAM Profile Saved] -> /kaggle/working/reports/vram_profile_electronics.png")


## Cell 8c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Electronics
- Trực quan hóa tiến trình hội tụ và quỹ đạo cải thiện Validation NDCG@20 trên Amazon Electronics (~1.7M tương tác).


In [ ]:
# # Cell 8c: Learning Dynamics & Convergence Profiles — Amazon Electronics
# fig, ax1 = plt.subplots(figsize=(10, 4.8), dpi=150)

# log_elec = STAIR5_V52_CONFIGS['electronics']['log']
# bpr_e, cl_e, tot_e = parse_training_losses(log_elec, telemetry_elec)
# val_ndcg_e = parse_valid_metric(log_elec, 'NDCG@20')

# if tot_e:
#     x_tot, y_tot = zip(*tot_e)
#     ax1.plot(x_tot, y_tot, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·KPE)')
# if bpr_e:
#     x_bpr, y_bpr = zip(*bpr_e)
#     ax1.plot(x_bpr, y_bpr, color='#2ca02c', lw=1.5, ls='--', alpha=0.8, label='Pure BPR Loss')
# if cl_e:
#     x_cl, y_cl = zip(*cl_e)
#     ax1.plot(x_cl, y_cl, color='#d62728', lw=1.2, ls=':', alpha=0.7, label='Raw KPE Loss')

# ax1.set_xlabel('Epoch', fontsize=10)
# ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
# ax1.tick_params(axis='y', labelcolor='#1f77b4')
# ax1.grid(True, linestyle='--', alpha=0.4)

# if val_ndcg_e:
#     ax2 = ax1.twinx()
#     x_val, y_val = zip(*val_ndcg_e)
#     ax2.plot(x_val, y_val, color='#9467bd', lw=2.2, marker='^', ms=3, label='Validation NDCG@20')
#     ax2.set_ylabel('Validation NDCG@20', color='#9467bd', fontsize=10)
#     ax2.tick_params(axis='y', labelcolor='#9467bd')
#     if 'electronics' in BASELINE_REF:
#         ax2.axhline(BASELINE_REF['electronics']['NDCG@20'], color='gray', ls='--', lw=1.2, label='Baseline STAIR (0.0303)')
#     if 'electronics' in V4_REF:
#         ax2.axhline(V4_REF['electronics']['NDCG@20'], color='#2ca02c', ls=':', lw=1.2, label='STAIR5-v4 (0.0316)')
#     lines1, labels1 = ax1.get_legend_handles_labels()
#     lines2, labels2 = ax2.get_legend_handles_labels()
#     ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=8.5, framealpha=0.9)
# else:
#     ax1.legend(loc='upper right', fontsize=9.0)

# plt.title('Amazon Electronics — Learning Dynamics & Convergence (STAIR5-v5.2 NLGCL-BPE)', fontsize=12, fontweight='bold')
# fig.tight_layout()
# plt.savefig('/kaggle/working/reports/learning_curve_electronics.png', dpi=300)
# plt.show()
# print("✅ [Learning Dynamics Saved] -> /kaggle/working/reports/learning_curve_electronics.png")


## Cell 9 📊 Bảng So sánh Tổng hợp Ablation Study Đa Phiên bản (Baseline vs v1 vs v2 vs v3 vs v4 vs v5.1)
- Lập bảng so sánh chi tiết tất cả các chỉ số đo lường: `Recall@10`, `Recall@20`, `NDCG@10`, `NDCG@20`, Peak VRAM, và % cải thiện tương đối.


In [ ]:
# Cell 9: Bảng Đối chuẩn Đa thế hệ Toàn diện (STAIR Baseline vs v4 vs v5.1)
from prettytable import PrettyTable

table = PrettyTable()
table.field_names = ['Dataset', 'Kiến trúc / Phiên bản', *TRACKED_METRICS, 'Δ vs Baseline (%)', 'Δ vs v4 (%)']

all_results = {
    'sports': globals().get('metrics_sports', {}),
    'baby': globals().get('metrics_baby', {}),
    'electronics': globals().get('metrics_elec', {}),
}

for ds in ('sports', 'baby', 'electronics'):
    ref_bl = BASELINE_REF[ds]['NDCG@20']
    ref_v4 = V4_REF[ds]['NDCG@20']
    runs = [
        ('STAIR Baseline (gốc)', BASELINE_REF[ds]),
        ('STAIR5-v4 (N-CSE)', V4_REF[ds]),
        ('STAIR5-v5.2 (NLGCL-BPE)', all_results[ds]),
    ]
    for label, vals in runs:
        if vals and all(m in vals for m in TRACKED_METRICS):
            d_bl = 100 * (vals['NDCG@20'] - ref_bl) / ref_bl
            d_v4 = 100 * (vals['NDCG@20'] - ref_v4) / ref_v4 if 'v5.1' in label else 0.0
            d_v4_str = f"{d_v4:+.2f}%" if 'v5.1' in label else "-"
            table.add_row([ds.upper(), label, *[f"{vals[m]:.4f}" for m in TRACKED_METRICS], f"{d_bl:+.2f}%", d_v4_str])
        else:
            table.add_row([ds.upper(), label, *['Chờ chạy']*4, '-', '-'])

print(table)


## Cell 10 📦 Đóng gói Lưu trữ Toàn bộ Minh chứng, Logs, Manifests & Checkpoints
- Nén toàn bộ kết quả, biểu đồ PNG, logs, telemetry và manifest JSON để nghiệm thu khóa luận tốt nghiệp.


In [ ]:
# Cell 10: Đóng gói Lưu trữ Toàn bộ Minh chứng (Logs, Checkpoints, Telemetry, Reports)
import os, shutil, subprocess, json, time

archive_dir = '/kaggle/working/stair5_v52_artifacts'
os.makedirs(archive_dir, exist_ok=True)

# 1. Thu thập logs và telemetry
target_dirs = [
    '/kaggle/working/logs/stair5_v52',
    '/kaggle/working/reports',
    '/kaggle/working/runs/stair5_v52',
]
for src in target_dirs:
    if os.path.exists(src):
        dst = os.path.join(archive_dir, os.path.basename(src))
        shutil.copytree(src, dst, dirs_exist_ok=True)

# 2. Tạo Manifest JSON tổng hợp
manifest = {
    "project": "STAIR5-v5.2 / NLGCL-BPE",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "datasets": {
        "sports": {
            "test_metrics": metrics_sports if 'metrics_sports' in locals() else {},
            "best_epoch": ep_sports if 'ep_sports' in locals() else None,
            "train_time_min": sports_time / 60 if 'sports_time' in locals() else None,
        },
        "baby": {
            "test_metrics": metrics_baby if 'metrics_baby' in locals() else {},
            "best_epoch": ep_baby if 'ep_baby' in locals() else None,
            "train_time_min": baby_time / 60 if 'baby_time' in locals() else None,
        },
        "electronics": {
            "test_metrics": metrics_elec if 'metrics_elec' in locals() else {},
            "best_epoch": ep_elec if 'ep_elec' in locals() else None,
            "train_time_min": electronics_time / 60 if 'electronics_time' in locals() else None,
        }
    }
}

manifest_path = os.path.join(archive_dir, "stair5_v52_manifest.json")
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# 3. Nén file zip hoàn chỉnh
zip_path = '/kaggle/working/stair5_v52_complete_artifacts.zip'
shutil.make_archive('/kaggle/working/stair5_v52_complete_artifacts', 'zip', archive_dir)

print("=" * 80)
print(f"📦 ĐÓNG GÓI HOÀN TẤT THÀNH CÔNG!")
print(f"  * Thư mục Artifacts : {archive_dir}")
print(f"  * File Nén Tổng Hợp : {zip_path} ({os.path.getsize(zip_path)/(1024**2):.2f} MB)")
print("=" * 80)
